# Walmart Store Sales Forecasting: Temporal Fusion Transformer (TFT)

##  TFT architecture

TFT is the "everything model" of forecasting: where DLinear/N-BEATS/PatchTST read only the target's history, TFT was built to fuse three distinct input types, each with its own pathway:

1. **Static covariates** (store Type, Size): encoded once per series and injected everywhere: they condition the variable selection, the sequence encoder and the attention.
2. **Known-future inputs** (holiday calendar, week of year): available for both past and future, so the decoder can *anticipate* events instead of extrapolating them.
3. **Observed-past inputs** (the sales history itself, and optionally markdowns/CPI): known only up to the forecast origin.

The machinery, briefly:

- **Variable Selection Networks**: at every time step, a learned softmax weighs each input variable's usefulness; irrelevant inputs are gated toward zero. This is why TFT is called interpretable: the weights are readable feature importances.
- **LSTM encoder-decoder** handles local sequential patterns; **interpretable multi-head attention** on top captures long-range dependencies (which past weeks matter for which future weeks).
- **Gating (GRN blocks)** throughout lets the network skip any component it does not need, in theory letting a big model degrade gracefully on simple data.

**Why we expect it to lose here, and why running it is still the right experiment:** our EDA measured near-zero linear correlation for the macro covariates and a 1.9% markdown lift; N-BEATSx got nothing from calendar flags (2389 vs 2390) or static attributes (2444). TFT bets its parameter budget on exactly those inputs. On 91-week series with weak covariates, that budget is likely wasted vs N-BEATS' history-focused blocks. The value of this notebook is settling the covariate question with the heaviest available machinery, in both directions.

**Inherited unchanged:** zero-filled weekly grid, input 52 (91-week constraint, `start_padding_enabled=True`), raw target, evaluation protocol, MLflow structure, pipeline pattern with the fixed-horizon guard.

## Setup

In [ ]:
!pip install -q neuralforecast mlflow dagshub

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import neuralforecast
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE
from utilsforecast.preprocessing import fill_gaps

print('neuralforecast', neuralforecast.__version__, '| mlflow', mlflow.__version__)

In [ ]:
import dagshub
from kaggle_secrets import UserSecretsClient

dagshub.auth.add_app_token(UserSecretsClient().get_secret('DAGSHUB_TOKEN'))
dagshub.init(repo_owner='Nestor-Dzadzamia', repo_name='walmart-sales-forecasting', mlflow=True)

mlflow.set_experiment('TFT_Training')
exp = mlflow.get_experiment_by_name('TFT_Training')
assert exp is not None and str(exp.experiment_id) != '0', 'runs would land in the Default experiment'
print('tracking uri:', mlflow.get_tracking_uri())
print('experiment:', exp.name, '| id:', exp.experiment_id)

## Shared contract (copied verbatim from the EDA notebook)

In [ ]:
def load_and_merge(df, features, stores):
    out = df.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
    out = out.merge(stores, on='Store', how='left')
    out['Date'] = pd.to_datetime(out['Date'])
    return out.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

MD_COLS = [f"MarkDown{i}" for i in range(1, 6)]

def preprocess(df):
    out = df.copy()
    out[MD_COLS] = out[MD_COLS].fillna(0)
    out[["CPI", "Unemployment"]] = out.groupby("Store")[["CPI", "Unemployment"]].ffill()
    return out

def wmae(y_true, y_pred, is_holiday):
    w = np.where(is_holiday, 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)

def coldstart_fallback(df_train, df_test):
    train_pairs = set(zip(df_train.Store, df_train.Dept))
    mask = ~pd.Series(list(zip(df_test.Store, df_test.Dept)), index=df_test.index).isin(train_pairs)
    cold = df_test[mask].copy()
    recent = df_train[df_train['Date'] >= df_train['Date'].max() - pd.Timedelta(weeks=52)]
    med = recent.groupby(['Type', 'Dept'])['Weekly_Sales'].median()
    cold['y_fallback'] = [med.get((t, d), 0.0) for t, d in zip(cold['Type'], cold['Dept'])]
    cold['y_fallback'] = cold['y_fallback'].clip(lower=0)
    return cold

In [ ]:
path = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/'

train = pd.read_csv(path + 'train.csv.zip')
test = pd.read_csv(path + 'test.csv.zip')
features = pd.read_csv(path + 'features.csv.zip')
stores = pd.read_csv(path + 'stores.csv')

df = load_and_merge(train, features, stores)
df_test = load_and_merge(test, features, stores)
print('train merged:', df.shape, '| test merged:', df_test.shape)

In [ ]:
with mlflow.start_run(run_name='TFT_Cleaning'):
    mlflow.log_param('markdown_fill', '0')
    mlflow.log_param('cpi_unemployment_fill', 'ffill_per_store')
    mlflow.log_metric('markdown_nan_before', int(df[MD_COLS].isna().sum().sum()))

    df_clean = preprocess(df)

    mlflow.log_metric('markdown_nan_after', int(df_clean[MD_COLS].isna().sum().sum()))
    mlflow.log_metric('train_rows', len(df_clean))

print(df_clean.shape)

In [ ]:
VALIDATION_START = df_test['Date'].min() - pd.Timedelta(weeks=52)
VALIDATION_END = VALIDATION_START + pd.Timedelta(weeks=39)

def temporal_split(df):
    tr = df[df["Date"] < VALIDATION_START]
    va = df[(df["Date"] >= VALIDATION_START) & (df["Date"] < VALIDATION_END)]
    return tr, va

tr, va = temporal_split(df_clean)
print('train:', tr['Date'].max(), tr.shape)
print('val:  ', va['Date'].min(), '->', va['Date'].max(), va.shape)

##  Data preparation: weekly grid + covariates

Same grid and the same covariate frames as the N-BEATSx experiments, so the covariate comparison across architectures is apples to apples: future-known holiday calendar (per-holiday flags, pre-Christmas week, weeks-to-Christmas, week-of-year sin/cos) and static store attributes (Type one-hot, scaled Size).

In [ ]:
HORIZON = 39
FREQ = 'W-FRI'

HOLIDAY_WEEKS = {
    'is_superbowl_week': ['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08'],
    'is_laborday_week': ['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06'],
    'is_thanksgiving_week': ['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29'],
    'is_christmas_week': ['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27'],
}
CHRISTMAS_FRIDAYS = pd.to_datetime(HOLIDAY_WEEKS['is_christmas_week'])

def add_calendar_exog(frame):
    out = frame.copy()
    ds = pd.to_datetime(out['ds'])
    for col, dates in HOLIDAY_WEEKS.items():
        out[col] = ds.isin(pd.to_datetime(dates)).astype(float)
    out['is_pre_christmas_week'] = ds.isin(CHRISTMAS_FRIDAYS - pd.Timedelta(weeks=1)).astype(float)
    idx = np.clip(np.searchsorted(CHRISTMAS_FRIDAYS.values, ds.values), 0, len(CHRISTMAS_FRIDAYS) - 1)
    nxt = CHRISTMAS_FRIDAYS.values[idx]
    out['weeks_to_christmas'] = np.clip((nxt - ds.values).astype('timedelta64[D]').astype(float) / 7.0, 0, None) / 52.0
    woy = ds.dt.isocalendar().week.astype(float)
    out['woy_sin'] = np.sin(2 * np.pi * woy / 52.0)
    out['woy_cos'] = np.cos(2 * np.pi * woy / 52.0)
    return out

FUTR_COLS = ['is_superbowl_week', 'is_laborday_week', 'is_thanksgiving_week',
             'is_christmas_week', 'is_pre_christmas_week', 'weeks_to_christmas',
             'woy_sin', 'woy_cos']

def to_long(frame):
    out = frame[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
    out['unique_id'] = out['Store'].astype(str) + '_' + out['Dept'].astype(str)
    out = out.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    return out[['unique_id', 'ds', 'y']].sort_values(['unique_id', 'ds']).reset_index(drop=True)

tr_long = fill_gaps(to_long(tr), freq=FREQ, start='per_serie', end='global')
n_gap_rows = int(tr_long['y'].isna().sum())
tr_long['y'] = tr_long['y'].fillna(0.0)
tr_long_exog = add_calendar_exog(tr_long)

stat = df_clean[['Store', 'Dept', 'Type', 'Size']].drop_duplicates(['Store', 'Dept']).copy()
stat['unique_id'] = stat['Store'].astype(str) + '_' + stat['Dept'].astype(str)
static_df = pd.DataFrame({
    'unique_id': stat['unique_id'].values,
    'type_A': (stat['Type'] == 'A').astype(float).values,
    'type_B': (stat['Type'] == 'B').astype(float).values,
    'size_scaled': (stat['Size'] / stat['Size'].max()).values,
})
STATIC_COLS = ['type_A', 'type_B', 'size_scaled']

def make_futr_df(last_ds, ids):
    future_ds = pd.date_range(last_ds + pd.Timedelta(weeks=1), periods=HORIZON, freq=FREQ)
    base = pd.DataFrame({'unique_id': np.repeat(ids, HORIZON),
                         'ds': np.tile(future_ds, len(ids))})
    return add_calendar_exog(base)

va_eval = va[['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday']].copy()
va_eval['unique_id'] = va_eval['Store'].astype(str) + '_' + va_eval['Dept'].astype(str)

with mlflow.start_run(run_name='TFT_Data_Prep'):
    mlflow.log_param('format', 'long (unique_id, ds, y) + calendar exog')
    mlflow.log_param('futr_exog', ', '.join(FUTR_COLS))
    mlflow.log_param('static_exog', ', '.join(STATIC_COLS))
    mlflow.log_metric('n_series_train', tr_long['unique_id'].nunique())
    mlflow.log_metric('gap_rows_zero_filled', n_gap_rows)

print('series:', tr_long['unique_id'].nunique(), '| rows:', len(tr_long), f'({n_gap_rows} gap rows zero-filled)')
print('static_df:', static_df.shape, '| validation rows:', len(va_eval))

##  Evaluation protocol and the run helper

Identical to the rest of the DL track. TFT-specific sizing: `hidden_size=64` and `windows_batch_size=256` keep run times sane on a T4 (TFT's LSTM + attention + gating stack is an order of magnitude heavier per step than N-BEATS).

In [ ]:
def train_and_eval(run_name, hidden_size=64, scaler_type='robust', use_futr=False,
                   use_static=False, input_size=52, max_steps=1000,
                   learning_rate=1e-3, batch_size=32, seed=42):
    kwargs = dict(h=HORIZON, input_size=input_size, hidden_size=hidden_size,
                  loss=MAE(), scaler_type=scaler_type, learning_rate=learning_rate,
                  max_steps=max_steps, batch_size=batch_size, windows_batch_size=256,
                  start_padding_enabled=True, random_seed=seed, logger=False, devices=1)
    if use_futr:
        kwargs['futr_exog_list'] = FUTR_COLS
    if use_static:
        kwargs['stat_exog_list'] = STATIC_COLS

    model = TFT(**kwargs)
    fit_df = tr_long_exog if use_futr else tr_long
    nf = NeuralForecast(models=[model], freq=FREQ)
    nf.fit(df=fit_df, static_df=static_df if use_static else None)

    if use_futr:
        futr = make_futr_df(tr_long['ds'].max(), fit_df['unique_id'].unique())
        preds = nf.predict(futr_df=futr)
    else:
        preds = nf.predict()
    if 'unique_id' not in preds.columns:
        preds = preds.reset_index()
    pred_col = [c for c in preds.columns if c not in ('unique_id', 'ds')][0]
    preds = preds.rename(columns={pred_col: 'y_hat'})

    merged = va_eval.merge(preds[['unique_id', 'ds', 'y_hat']],
                           left_on=['unique_id', 'Date'], right_on=['unique_id', 'ds'], how='left')
    coverage = float(merged['y_hat'].notna().mean())
    merged['y_hat'] = merged['y_hat'].fillna(0).clip(lower=0)

    score = float(wmae(merged['Weekly_Sales'], merged['y_hat'], merged['IsHoliday']))
    err = np.abs(merged['Weekly_Sales'] - merged['y_hat'])
    hol = merged['IsHoliday'].astype(bool)
    metrics = {'val_wmae': score,
               'val_mae': float(err.mean()),
               'val_mae_holiday': float(err[hol].mean()),
               'val_mae_regular': float(err[~hol].mean()),
               'pred_coverage': coverage}
    params = {'hidden_size': hidden_size, 'scaler_type': scaler_type, 'use_futr': use_futr,
              'use_static': use_static, 'input_size': input_size, 'max_steps': max_steps,
              'learning_rate': learning_rate, 'batch_size': batch_size, 'seed': seed,
              'horizon': HORIZON, 'loss': 'MAE'}

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        for k, v in metrics.items():
            mlflow.log_metric(k, v)

    wmae_str = f'{score:,.2f}'
    print(f'{run_name}: WMAE={wmae_str} | MAE={metrics["val_mae"]:,.2f} | '
          f'holiday MAE={metrics["val_mae_holiday"]:,.2f} | coverage={coverage:.1%}')
    return {'run': run_name, **params, **metrics}, nf, merged

results = []
merged_map = {}

## Baseline

TFT with no covariates at all: pure history through the LSTM + attention machinery, robust per-window scaling (the neuralforecast default for TFT), hidden 64. The question: what does the heavy architecture deliver before its signature inputs even arrive? Reference: N-BEATS 2342.47.

In [ ]:
res, _, m = train_and_eval('TFT_Baseline')
results.append(res); merged_map[res['run']] = m
print('references: N-BEATS 2342.47 | PatchTST 2529.41 | DLinear 3385.80 | LightGBM 1811')

## Experiment: scaling

Track history: DLinear improved with an explicit scaler, N-BEATS was hurt by one, PatchTST collapsed without RevIN. TFT defaults to robust per-window scaling; one run with identity shows whether the gating machinery survives raw magnitudes.

In [ ]:
res, _, m = train_and_eval('TFT_Scaler_identity', scaler_type='identity')
results.append(res); merged_map[res['run']] = m

fam = [r for r in results if r['run'] in ('TFT_Baseline', 'TFT_Scaler_identity')]
best_scaler = min(fam, key=lambda r: r['val_wmae'])['scaler_type']
print('carrying forward scaler_type =', best_scaler)

##  Experiment: capacity

In [ ]:
res, _, m = train_and_eval('TFT_Hidden_128', hidden_size=128, scaler_type=best_scaler)
results.append(res); merged_map[res['run']] = m

fam = [r for r in results if r['scaler_type'] == best_scaler and not r['use_futr'] and not r['use_static']]
best_hidden = int(min(fam, key=lambda r: r['val_wmae'])['hidden_size'])
print('carrying forward hidden_size =', best_hidden)

##  The decisive experiment: covariates through TFT's native machinery

N-BEATSx bolted covariates onto residual blocks and got nothing (2389 vs 2390 without; static made it worse). TFT was *designed* for them: variable selection, static encoders, known-future pathway. If these two runs do not beat the TFT baseline, the covariate question is closed for this dataset with the strongest possible evidence.

In [ ]:
res, _, m = train_and_eval('TFT_Calendar', scaler_type=best_scaler,
                           hidden_size=best_hidden, use_futr=True)
results.append(res); merged_map[res['run']] = m

In [ ]:
res, _, m = train_and_eval('TFT_Calendar_Static', scaler_type=best_scaler,
                           hidden_size=best_hidden, use_futr=True, use_static=True)
results.append(res); merged_map[res['run']] = m

##  Results

In [ ]:
results_df = pd.DataFrame(results).sort_values('val_wmae').reset_index(drop=True)
cols = ['run', 'hidden_size', 'scaler_type', 'use_futr', 'use_static',
        'val_wmae', 'val_mae', 'val_mae_holiday', 'val_mae_regular', 'pred_coverage']
display(results_df[cols])
print('references: N-BEATS 2342.47 | PatchTST 2529.41 | DLinear 3385.80 | LightGBM 1811')

##  Error analysis of the best run

In [ ]:
best_run_name = results_df.iloc[0]['run']
best = merged_map[best_run_name]
print('best run:', best_run_name)

by_week = best.groupby('Date').apply(lambda g: wmae(g['Weekly_Sales'], g['y_hat'], g['IsHoliday']))
hol_weeks = best.loc[best['IsHoliday'].astype(bool), 'Date'].unique()

fig, ax = plt.subplots(figsize=(13, 4))
by_week.plot(ax=ax)
for d in hol_weeks:
    ax.axvline(d, color='red', alpha=0.3, linestyle='--')
ax.set_title(f'{best_run_name}: WMAE per validation week (red = 5x-weighted holiday weeks)')
ax.set_ylabel('WMAE')
plt.show()

nb_holiday = 3141.40
tft_holiday = float(results_df.iloc[0]['val_mae_holiday'])
print(f'holiday MAE: DLinear 4605 -> N-BEATS 3141 -> PatchTST 3711 -> TFT {tft_holiday:,.0f} '
      f'({(tft_holiday / nb_holiday - 1) * 100:+.1f}% vs N-BEATS)')

##  Final model: retrain on full history

In [ ]:
best_row = results_df.iloc[0]
best_cfg = {'hidden_size': int(best_row['hidden_size']), 'scaler_type': str(best_row['scaler_type']),
            'use_futr': bool(best_row['use_futr']), 'use_static': bool(best_row['use_static']),
            'input_size': int(best_row['input_size']), 'max_steps': int(best_row['max_steps'])}
best_val_wmae = float(best_row['val_wmae'])
print('final config:', best_cfg, '| validated WMAE:', round(best_val_wmae, 2))

full_long = fill_gaps(to_long(df_clean), freq=FREQ, start='per_serie', end='global')
full_long['y'] = full_long['y'].fillna(0.0)
full_long_exog = add_calendar_exog(full_long)

kwargs = dict(h=HORIZON, input_size=best_cfg['input_size'], hidden_size=best_cfg['hidden_size'],
              loss=MAE(), scaler_type=best_cfg['scaler_type'], learning_rate=1e-3,
              max_steps=best_cfg['max_steps'], batch_size=32, windows_batch_size=256,
              start_padding_enabled=True, random_seed=42, logger=False, devices=1)
if best_cfg['use_futr']:
    kwargs['futr_exog_list'] = FUTR_COLS
if best_cfg['use_static']:
    kwargs['stat_exog_list'] = STATIC_COLS

final_model = TFT(**kwargs)
fit_full = full_long_exog if best_cfg['use_futr'] else full_long
nf_final = NeuralForecast(models=[final_model], freq=FREQ)
nf_final.fit(df=fit_full, static_df=static_df if best_cfg['use_static'] else None)
print('final model trained on', full_long['unique_id'].nunique(), 'series,', len(full_long), 'rows')

In [ ]:
if best_cfg['use_futr']:
    futr = make_futr_df(full_long['ds'].max(), fit_full['unique_id'].unique())
    preds_test = nf_final.predict(futr_df=futr)
else:
    preds_test = nf_final.predict()
if 'unique_id' not in preds_test.columns:
    preds_test = preds_test.reset_index()
pred_col = [c for c in preds_test.columns if c not in ('unique_id', 'ds')][0]
preds_test = preds_test.rename(columns={pred_col: 'y_hat'})

test_pred = df_test[['Store', 'Dept', 'Date']].copy()
test_pred['unique_id'] = test_pred['Store'].astype(str) + '_' + test_pred['Dept'].astype(str)
test_pred = test_pred.merge(preds_test[['unique_id', 'ds', 'y_hat']],
                            left_on=['unique_id', 'Date'], right_on=['unique_id', 'ds'], how='left')
print('test rows without model forecast (expect 36 cold-start rows):', int(test_pred['y_hat'].isna().sum()))

cold = coldstart_fallback(df_clean, df_test)
test_pred = test_pred.merge(cold[['Store', 'Dept', 'Date', 'y_fallback']],
                            on=['Store', 'Dept', 'Date'], how='left')
test_pred['Weekly_Sales'] = test_pred['y_fallback'].combine_first(test_pred['y_hat']).fillna(0).clip(lower=0)

submission = test_pred[['Store', 'Dept', 'Date']].copy()
submission['Id'] = (submission['Store'].astype(str) + '_' + submission['Dept'].astype(str)
                    + '_' + submission['Date'].dt.strftime('%Y-%m-%d'))
submission['Weekly_Sales'] = test_pred['Weekly_Sales']
submission = submission[['Id', 'Weekly_Sales']].sort_values('Id').reset_index(drop=True)
assert len(submission) == len(test), f'row mismatch: {len(submission)} vs {len(test)}'
submission.to_csv('submission_tft.csv', index=False)
print(submission.shape)
submission.head()

## Pipeline artifact

Same verified pattern as the rest of the track, covariate-aware, with the fixed-horizon date-range guard (raises a clear error for dates outside the 39-week horizon instead of silently returning fallback values).

In [ ]:
nf_final.save(path='nf_tft', overwrite=True, save_dataset=True)
df_clean.to_parquet('train_clean.parquet')
features.to_parquet('features.parquet')
stores.to_parquet('stores.parquet')

class TFTPipeline(mlflow.pyfunc.PythonModel):
    use_futr = bool(best_cfg['use_futr'])

    def load_context(self, context):
        from neuralforecast import NeuralForecast
        self.nf = NeuralForecast.load(context.artifacts['nf_model'])
        self.train_clean = pd.read_parquet(context.artifacts['train_clean'])
        self.features = pd.read_parquet(context.artifacts['features'])
        self.stores = pd.read_parquet(context.artifacts['stores'])

    def predict(self, context, model_input):
        clean = preprocess(load_and_merge(model_input.copy(), self.features, self.stores))

        if self.use_futr:
            ids = (self.train_clean['Store'].astype(str) + '_'
                   + self.train_clean['Dept'].astype(str)).unique()
            dates = pd.to_datetime(sorted(clean['Date'].unique()))
            futr = pd.DataFrame({'unique_id': np.repeat(ids, len(dates)),
                                 'ds': np.tile(dates.values, len(ids))})
            preds = self.nf.predict(futr_df=add_calendar_exog(futr))
        else:
            preds = self.nf.predict()
        if 'unique_id' not in preds.columns:
            preds = preds.reset_index()
        pred_col = [c for c in preds.columns if c not in ('unique_id', 'ds')][0]
        preds = preds.rename(columns={pred_col: 'y_hat'})

        horizon_ds = set(pd.to_datetime(preds['ds'].unique()))
        requested = set(pd.to_datetime(clean['Date'].unique()))
        outside = sorted(d for d in requested if d not in horizon_ds)
        if outside:
            raise ValueError(
                f'{len(outside)} requested dates fall outside the fixed 39-week forecast horizon '
                f'(first: {outside[0].date()}). This pipeline forecasts only the 39 weeks '
                f'immediately after its training history; retrain to forecast other ranges.')

        out = clean[['Store', 'Dept', 'Date']].copy()
        out['unique_id'] = out['Store'].astype(str) + '_' + out['Dept'].astype(str)
        out = out.merge(preds[['unique_id', 'ds', 'y_hat']],
                        left_on=['unique_id', 'Date'], right_on=['unique_id', 'ds'], how='left')
        cold = coldstart_fallback(self.train_clean, clean)
        out = out.merge(cold[['Store', 'Dept', 'Date', 'y_fallback']],
                        on=['Store', 'Dept', 'Date'], how='left')
        out['Weekly_Sales'] = out['y_fallback'].combine_first(out['y_hat']).fillna(0).clip(lower=0)
        return out[['Store', 'Dept', 'Date', 'Weekly_Sales']]

with mlflow.start_run(run_name='TFT_FINAL_MODEL') as run:
    mlflow.log_params(best_cfg)
    mlflow.log_param('trained_on', 'full_history_143_weeks')
    mlflow.log_param('coldstart', 'shared coldstart_fallback')
    mlflow.log_metric('validation_wmae', best_val_wmae)
    model_info = mlflow.pyfunc.log_model(
        artifact_path='model',
        python_model=TFTPipeline(),
        artifacts={'nf_model': 'nf_tft',
                   'train_clean': 'train_clean.parquet',
                   'features': 'features.parquet',
                   'stores': 'stores.parquet'},
        pip_requirements=[f'neuralforecast=={neuralforecast.__version__}', 'pandas', 'pyarrow'],
    )
    final_run_id = run.info.run_id

print('final run id:', final_run_id)
print('model uri:', model_info.model_uri)

In [ ]:
loaded = mlflow.pyfunc.load_model(model_info.model_uri)
pipe_out = loaded.predict(test)

check = test_pred[['Store', 'Dept', 'Date', 'Weekly_Sales']].merge(
    pipe_out, on=['Store', 'Dept', 'Date'], suffixes=('_nb', '_pipe'))
assert len(check) == len(test), 'pipeline did not return every test row'
max_diff = float((check['Weekly_Sales_nb'] - check['Weekly_Sales_pipe']).abs().max())
print('max abs diff notebook vs pipeline:', max_diff)
assert max_diff < 1.0
print('PIPELINE SMOKE TEST PASSED: raw test.csv in, final predictions out')